In [1]:
import os
import warnings
import tqdm
import pandas as pd
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)

In [2]:
%load_ext autoreload
%autoreload 2
import socceraction.vaep.features as fs
import socceraction.vaep.labels as lab

In [3]:
datafolder = "../data-fifa"
spadl_h5 = os.path.join(datafolder, "spadl-statsbomb.h5")
features_h5 = os.path.join(datafolder, "features.h5")
labels_h5 = os.path.join(datafolder, "labels.h5")
predictions_h5 = os.path.join(datafolder, "predictions.h5")

In [4]:
games = pd.read_hdf(spadl_h5, "games")
print("nb of games:", len(games))

# note: only for the purpose of this example and due to the small dataset,
# we use the same data for training and evaluation
traingames = games
testgames = games

nb of games: 64


In [5]:
testgames

,game_id,season_id,competition_id,competition_stage,game_day,game_date,home_team_id,away_team_id,home_score,away_score,venue,referee,competition_name,country_name,competition_gender,season_name,home_team_name,away_team_name
0,7548,3,43,Group Stage,2,2018-06-22 14:00:00,781,795,2,0,Saint-Petersburg Stadium,Björn Kuipers,FIFA World Cup,International,male,2018,Brazil,Costa Rica
1,7578,3,43,Group Stage,1,2018-06-15 14:00:00,774,783,0,1,\tEkaterinburg Arena,Björn Kuipers,FIFA World Cup,International,male,2018,Egypt,Uruguay
2,7553,3,43,Group Stage,2,2018-06-23 17:00:00,791,794,1,2,\tRostov Arena,Milorad Mažić,FIFA World Cup,International,male,2018,South Korea,Mexico
3,7544,3,43,Group Stage,2,2018-06-20 17:00:00,783,799,1,0,\tRostov Arena,Clément Turpin,FIFA World Cup,International,male,2018,Uruguay,Saudi Arabia
4,7536,3,43,Group Stage,1,2018-06-18 17:00:00,782,798,3,0,\tOlimpiyskiy Stadion Fisht,Janny Sikazwe,FIFA World Cup,International,male,2018,Belgium,Panama
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,7540,3,43,Group Stage,2,2018-06-19 20:00:00,796,774,3,1,Saint-Petersburg Stadium,Enrique Cáceres,FIFA World Cup,International,male,2018,Russia,Egypt
60,8652,3,43,Quarter-finals,5,2018-07-07 20:00:00,796,785,2,2,\tOlimpiyskiy Stadion Fisht,Sandro Ricci,FIFA World Cup,International,male,2018,Russia,Croatia
61,7563,3,43,Group Stage,3,2018-06-26 16:00:00,776,771,0,0,Stadion Luzhniki,Sandro Ricci,FIFA World Cup,International,male,2018,Denmark,France
62,7556,3,43,Group Stage,2,2018-06-24 17:00:00,778,787,2,2,\tEkaterinburg Arena,Gianluca Rocchi,FIFA World Cup,International,male,2018,Japan,Senegal


In [6]:
# 1. Select feature set X
xfns = [
    fs.actiontype,
    fs.actiontype_onehot,
    #fs.bodypart,
    fs.bodypart_onehot,
    fs.result,
    fs.result_onehot,
    fs.goalscore,
    fs.startlocation,
    fs.endlocation,
    fs.movement,
    fs.space_delta,
    fs.startpolar,
    fs.endpolar,
    fs.team,
    #fs.time,
    fs.time_delta,
    #fs.actiontype_result_onehot
]
nb_prev_actions = 3

Xcols = fs.feature_column_names(xfns, nb_prev_actions)

def getXY(games,Xcols):
    # generate the columns of the selected feature
    X = []
    for game_id in tqdm.tqdm(games.game_id, desc="Selecting features"):
        Xi = pd.read_hdf(features_h5, f"game_{game_id}")
        X.append(Xi[Xcols])
    X = pd.concat(X).reset_index(drop=True)

    # 2. Select label Y
    Ycols = ["scores","concedes"]
    Y = []
    for game_id in tqdm.tqdm(games.game_id, desc="Selecting label"):
        Yi = pd.read_hdf(labels_h5, f"game_{game_id}")
        Y.append(Yi[Ycols])
    Y = pd.concat(Y).reset_index(drop=True)
    return X, Y

X, Y = getXY(traingames,Xcols)
print("X:", list(X.columns))
print("Y:", list(Y.columns))

Selecting label: 100%|██████████| 64/64 [00:00<00:00, 220.02it/s]

X: ['type_id_a0', 'type_id_a1', 'type_id_a2', 'type_pass_a0', 'type_cross_a0', 'type_throw_in_a0', 'type_freekick_crossed_a0', 'type_freekick_short_a0', 'type_corner_crossed_a0', 'type_corner_short_a0', 'type_take_on_a0', 'type_foul_a0', 'type_tackle_a0', 'type_interception_a0', 'type_shot_a0', 'type_shot_penalty_a0', 'type_shot_freekick_a0', 'type_keeper_save_a0', 'type_keeper_claim_a0', 'type_keeper_punch_a0', 'type_keeper_pick_up_a0', 'type_clearance_a0', 'type_bad_touch_a0', 'type_non_action_a0', 'type_dribble_a0', 'type_goalkick_a0', 'type_pass_a1', 'type_cross_a1', 'type_throw_in_a1', 'type_freekick_crossed_a1', 'type_freekick_short_a1', 'type_corner_crossed_a1', 'type_corner_short_a1', 'type_take_on_a1', 'type_foul_a1', 'type_tackle_a1', 'type_interception_a1', 'type_shot_a1', 'type_shot_penalty_a1', 'type_shot_freekick_a1', 'type_keeper_save_a1', 'type_keeper_claim_a1', 'type_keeper_punch_a1', 'type_keeper_pick_up_a1', 'type_clearance_a1', 'type_bad_touch_a1', 'type_non_action_

In [7]:
X.shape,Y.shape

((128937, 151), (128937, 2))

In [8]:
X

,type_id_a0,type_id_a1,type_id_a2,type_pass_a0,type_cross_a0,type_throw_in_a0,type_freekick_crossed_a0,type_freekick_short_a0,type_corner_crossed_a0,type_corner_short_a0,...,end_dist_to_goal_a0,end_angle_to_goal_a0,end_dist_to_goal_a1,end_angle_to_goal_a1,end_dist_to_goal_a2,end_angle_to_goal_a2,team_1,team_2,time_delta_1,time_delta_2
0,0,0,0,True,False,False,False,False,False,False,...,87.354001,0.004927,87.354001,0.004927,87.354001,0.004927,True,True,0.0,0.0
1,21,0,0,False,False,False,False,False,False,False,...,84.838075,0.055832,87.354001,0.004927,87.354001,0.004927,True,True,2.0,2.0
2,0,21,0,True,False,False,False,False,False,False,...,32.550722,0.572928,84.838075,0.055832,87.354001,0.004927,True,True,2.0,4.0
3,0,0,21,True,False,False,False,False,False,False,...,76.671868,0.209214,79.626829,0.223458,20.838993,0.229180,False,False,3.0,5.0
4,21,0,0,False,False,False,False,False,False,False,...,75.023036,0.155515,76.671868,0.209214,79.626829,0.223458,True,False,1.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
128932,0,9,7,True,False,False,False,False,False,False,...,13.863228,0.795235,16.142562,1.406072,16.142562,1.406072,False,True,2.0,2.0
128933,21,0,9,False,False,False,False,False,False,False,...,15.727082,0.753421,13.863228,0.795235,16.142562,1.406072,True,False,1.0,3.0
128934,0,21,0,True,False,False,False,False,False,False,...,11.628859,1.181669,15.727082,0.753421,13.863228,0.795235,True,True,0.0,1.0
128935,11,0,21,False,False,False,False,False,False,False,...,3.529114,1.570796,11.628859,1.181669,15.727082,0.753421,True,True,1.0,1.0


In [9]:
Y

,scores,concedes
0,False,False
1,False,False
2,False,False
3,False,False
4,False,False
...,...,...
128932,True,False
128933,True,False
128934,True,False
128935,True,False


In [22]:
# 3. train classifiers F(X) = Y
import xgboost
import catboost
Y_hat = pd.DataFrame()
models = {}
for col in list(Y.columns):
    model = catboost.CatBoostClassifier()
    model.fit(X, Y[col])
    models[col] = model
model

Learning rate set to 0.082047
0:	learn: 0.5225217	total: 156ms	remaining: 2m 35s
1:	learn: 0.4007154	total: 178ms	remaining: 1m 28s
2:	learn: 0.3096112	total: 199ms	remaining: 1m 6s
3:	learn: 0.2394887	total: 220ms	remaining: 54.7s
4:	learn: 0.1883602	total: 242ms	remaining: 48.1s
5:	learn: 0.1530675	total: 272ms	remaining: 45s
6:	learn: 0.1281892	total: 298ms	remaining: 42.3s
7:	learn: 0.1098956	total: 325ms	remaining: 40.3s
8:	learn: 0.0956320	total: 348ms	remaining: 38.3s
9:	learn: 0.0857088	total: 374ms	remaining: 37s
10:	learn: 0.0772978	total: 399ms	remaining: 35.9s
11:	learn: 0.0710094	total: 423ms	remaining: 34.8s
12:	learn: 0.0658546	total: 445ms	remaining: 33.8s
13:	learn: 0.0618745	total: 466ms	remaining: 32.8s
14:	learn: 0.0589968	total: 488ms	remaining: 32s
15:	learn: 0.0565342	total: 509ms	remaining: 31.3s
16:	learn: 0.0543457	total: 530ms	remaining: 30.6s
17:	learn: 0.0527572	total: 550ms	remaining: 30s
18:	learn: 0.0516389	total: 572ms	remaining: 29.5s
19:	learn: 0.0506

In [23]:
from sklearn.metrics import brier_score_loss, roc_auc_score, log_loss

testX, testY = X, Y

def evaluate(y, y_hat):
    p = sum(y) / len(y)
    base = [p] * len(y)
    brier = brier_score_loss(y, y_hat)
    print(f"  Brier score: %.5f (%.5f)" % (brier, brier / brier_score_loss(y, base)))
    ll = log_loss(y, y_hat)
    print(f"  log loss score: %.5f (%.5f)" % (ll, ll / log_loss(y, base)))
    print(f"  ROC AUC: %.5f" % roc_auc_score(y, y_hat))

for col in testY.columns:
    Y_hat[col] = [p[1] for p in models[col].predict_proba(testX)]
    print(f"### Y: {col} ###")
    evaluate(testY[col], Y_hat[col])

### Y: scores ###
  Brier score: 0.00614 (0.59181)
  log loss score: 0.02964 (0.50871)
  ROC AUC: 0.93885
### Y: concedes ###
  Brier score: 0.00132 (0.49938)
  log loss score: 0.00716 (0.38869)
  ROC AUC: 0.97353


In [32]:
A = []
for game_id in tqdm.tqdm(games.game_id, "Loading game ids"):
    Ai = pd.read_hdf(spadl_h5, f"actions/game_{game_id}")
    A.append(Ai[["game_id"]])
A = pd.concat(A)
A = A.reset_index(drop=True)

# concatenate action game id rows with predictions and save per game
grouped_predictions = pd.concat([A, Y_hat], axis=1).groupby("game_id")
for k, df in tqdm.tqdm(grouped_predictions, desc="Saving predictions per game"):
    df = df.reset_index(drop=True)
    df[Y_hat.columns].to_hdf(predictions_h5, f"game_{int(k)}")

Saving predictions per game: 100%|██████████| 64/64 [00:00<00:00, 68.58it/s]


In [33]:
Y_hat

,scores,concedes
0,0.000259,0.002054
1,0.000718,0.002020
2,0.002524,0.001847
3,0.003820,0.020883
4,0.004918,0.000672
...,...,...
128932,0.521835,0.000156
128933,0.161773,0.000193
128934,0.243102,0.000588
128935,0.977680,0.000472


In [34]:
Y

,scores,concedes
0,False,False
1,False,False
2,False,False
3,False,False
4,False,False
...,...,...
128932,True,False
128933,True,False
128934,True,False
128935,True,False
